In [0]:
import requests
import time
from datetime import datetime, timedelta

# Target table that will be recreated from scratch
target_table_name = "finance_intelligence_hub.bronze.company_news_polygon"

# Full historical extraction range
start_date_str = "2026-07-03"
end_date_str = "2026-07-04"

# Start directly from the beginning date
current_date = datetime.strptime(start_date_str, "%Y-%m-%d").date()
end_date = datetime.strptime(end_date_str, "%Y-%m-%d").date()

# Polygon API configuration
api_key = "9dsWtWB7r0h5ZXqzfxVcdo1VoGmHSF1o"

grand_total_saved = 0
is_first_day = True

print("=" * 70)
print("WARNING: Existing table will be overwritten.")
print(f"Starting full historical extraction from {current_date} to {end_date}")
print("=" * 70)

# Main loop: process one day at a time
while current_date <= end_date:

    date_str = current_date.strftime("%Y-%m-%d")
    print(f"\nProcessing date: {date_str}")

    url = "https://api.polygon.io/v2/reference/news"

    params = {
        "published_utc.gte": f"{date_str}T00:00:00Z",
        "published_utc.lte": f"{date_str}T23:59:59Z",
        "order": "desc",
        "limit": 1000,
        "apiKey": api_key
    }

    day_news_count = 0
    page = 1
    all_records = []

    while url:

        response = requests.get(
            url,
            params=params if "apiKey" in params else None
        )

        if response.status_code == 200:

            data = response.json()
            results = data.get("results", [])

            if not results:
                break

            for news in results:

                title = news.get("title", "")
                description = news.get("description", "")
                published_date = news.get("published_utc", "")

                full_text = f"{title}. {description}".strip()

                tickers = news.get("tickers", [])

                related_tickers = ",".join(tickers) if tickers else "GENERAL"
                primary_ticker = tickers[0] if tickers else "GENERAL"

                all_records.append({

                    "ticker": primary_ticker,
                    "published_date": published_date,
                    "title": title,
                    "description": description,
                    "text_for_finbert": full_text,
                    "author": news.get("author", ""),
                    "source_url": news.get("article_url", ""),
                    "image_url": news.get("image_url", ""),
                    "publisher_name": news.get("publisher", {}).get("name", ""),
                    "publisher_url": news.get("publisher", {}).get("homepage_url", ""),
                    "related_tickers": related_tickers

                })

            day_news_count += len(results)

            print(
                f"   Page {page}: "
                f"{len(results)} news articles retrieved "
                f"(Daily total: {day_news_count})"
            )

            page += 1

            next_url = data.get("next_url")

            if next_url:
                url = f"{next_url}&apiKey={api_key}"
                params = {}
                time.sleep(12)
            else:
                url = None

        elif response.status_code == 429:

            print("Rate limit reached. Waiting 30 seconds...")
            time.sleep(30)
            continue

        else:

            print(
                f"Failed to retrieve news for {date_str}. "
                f"Status Code: {response.status_code}"
            )

            print("Retrying in 15 seconds...")
            time.sleep(15)
            continue

    # Save one day's data immediately
    if all_records:

        spark_df = spark.createDataFrame(all_records)

        if is_first_day:

            write_mode = "overwrite"
            is_first_day = False

            print(f"Table {target_table_name} overwritten successfully.")

        else:

            write_mode = "append"

        (
            spark_df.write
            .format("delta")
            .mode(write_mode)
            .option("mergeSchema", "true")
            .saveAsTable(target_table_name)
        )

        grand_total_saved += len(all_records)
        all_records.clear()

    print(f"Finished processing {date_str} successfully.")

    current_date += timedelta(days=1)

print("\n" + "=" * 70)
print("Historical extraction completed successfully.")
print(f"Total news articles saved: {grand_total_saved}")
print("=" * 70)